E-Commerce ETL Pipeline

This notebook implements the data extraction, transformation, and loading process using Apache Spark.

Source
- MariaDB

Processing
- Apache Spark
- Spark SQL

Target
- Hive

 Spark Session and Database Connection

Initialize Spark with Hive support and configure the JDBC connection to MariaDB.

In [1]:
import os
from pyspark.sql import SparkSession

# Path to the MariaDB JDBC driver (override with the JDBC_JAR environment variable)
JDBC_JAR = os.getenv("JDBC_JAR", "/home/student/Downloads/mariadb-java-client-3.3.3.jar")

spark = SparkSession.builder \
    .appName("Ecommerce_ETL") \
    .config("spark.jars", JDBC_JAR) \
    .enableHiveSupport() \
    .getOrCreate()

print("Spark started successfully")

Spark started successfully


In [2]:
import os

# Database settings are read from environment variables
DB_URL = os.getenv("DB_URL", "jdbc:mysql://localhost:3306/ecommerce?permitMysqlScheme")
DB_USER = os.getenv("DB_USER", "student")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")
DB_DRIVER = "org.mariadb.jdbc.Driver"

 Extract Data from MariaDB

Read the source tables from MariaDB using JDBC.

In [3]:
#Read the customers table using JDBC with an appropriate fetch size.
customers = spark.read \
    .format("jdbc") \
    .option("url", DB_URL) \
    .option("dbtable", "customers") \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", DB_DRIVER) \
    .option("fetchsize", "10000") \
    .load()

In [4]:
customers.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: date (nullable = true)



In [18]:
customers.show(5)

+-----------+----------+---------+--------------------+-------+--------------+-----------+
|customer_id|first_name|last_name|               email|country|          city|signup_date|
+-----------+----------+---------+--------------------+-------+--------------+-----------+
|          1|     Robin|    Heath|robin.heath1@exam...| Kuwait|  West Matthew| 2023-03-26|
|          2|     Sarah|     Long|sarah.long2@examp...|    UAE|    Nelsonside| 2022-10-29|
|          3|      Lori|    Green|lori.green3@examp...|  Qatar| Millerchester| 2021-10-17|
|          4|     April|   Sexton|april.sexton4@exa...| Kuwait|New Emilyburgh| 2022-01-03|
|          5|    Andrew|   Flores|andrew.flores5@ex...| Kuwait|    Kellyhaven| 2026-07-27|
+-----------+----------+---------+--------------------+-------+--------------+-----------+
only showing top 5 rows



In [5]:
customers.createOrReplaceTempView("customers")

In [6]:
spark.sql("""
SELECT *
FROM customers
LIMIT 5
""").show()

+-----------+----------+---------+--------------------+-------+--------------+-----------+
|customer_id|first_name|last_name|               email|country|          city|signup_date|
+-----------+----------+---------+--------------------+-------+--------------+-----------+
|          1|     Robin|    Heath|robin.heath1@exam...| Kuwait|  West Matthew| 2023-03-26|
|          2|     Sarah|     Long|sarah.long2@examp...|    UAE|    Nelsonside| 2022-10-29|
|          3|      Lori|    Green|lori.green3@examp...|  Qatar| Millerchester| 2021-10-17|
|          4|     April|   Sexton|april.sexton4@exa...| Kuwait|New Emilyburgh| 2022-01-03|
|          5|    Andrew|   Flores|andrew.flores5@ex...| Kuwait|    Kellyhaven| 2026-07-27|
+-----------+----------+---------+--------------------+-------+--------------+-----------+



In [7]:
#Read the products table using JDBC with an appropriate fetch size.
products = spark.read \
    .format("jdbc") \
    .option("url", DB_URL) \
    .option("dbtable", "products") \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", DB_DRIVER) \
    .option("fetchsize", "10000") \
    .load()

In [8]:
products.printSchema()

root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- stock_quantity: integer (nullable = true)



In [9]:
products.show(5, False)

+----------+-------------------------------------------+-----------+-------+--------------+
|product_id|product_name                               |category   |price  |stock_quantity|
+----------+-------------------------------------------+-----------+-------+--------------+
|1         |Decentralized explicit solution            |Home       |927.11 |526           |
|2         |Up-sized 3rdgeneration system engine       |Beauty     |690.18 |7             |
|3         |Open-source tertiary complexity            |Accessories|1237.17|376           |
|4         |Seamless holistic policy                   |Books      |822.53 |249           |
|5         |Reactive contextually-based functionalities|Clothing   |1443.08|832           |
+----------+-------------------------------------------+-----------+-------+--------------+
only showing top 5 rows



In [11]:
products.createOrReplaceTempView("products")

In [12]:
spark.sql("""
SELECT *
FROM products
LIMIT 5
""").show()

+----------+--------------------+-----------+-------+--------------+
|product_id|        product_name|   category|  price|stock_quantity|
+----------+--------------------+-----------+-------+--------------+
|         1|Decentralized exp...|       Home| 927.11|           526|
|         2|Up-sized 3rdgener...|     Beauty| 690.18|             7|
|         3|Open-source terti...|Accessories|1237.17|           376|
|         4|Seamless holistic...|      Books| 822.53|           249|
|         5|Reactive contextu...|   Clothing|1443.08|           832|
+----------+--------------------+-----------+-------+--------------+



In [13]:
#Read the orders table using JDBC with an appropriate fetch size and partitioning.
orders = spark.read \
    .format("jdbc") \
    .option("url", DB_URL) \
    .option("dbtable", "orders") \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", DB_DRIVER) \
    .option("fetchsize", "10000") \
    .option("partitionColumn", "order_id") \
    .option("lowerBound", "1") \
    .option("upperBound", "500000") \
    .option("numPartitions", "4") \
    .load()

In [14]:
orders.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- shipping_country: string (nullable = true)



In [15]:
orders.createOrReplaceTempView("orders")

In [16]:
spark.sql("""
SELECT *
FROM orders
LIMIT 5
""").show()

+--------+-----------+-------------------+---------+----------------+
|order_id|customer_id|         order_date|   status|shipping_country|
+--------+-----------+-------------------+---------+----------------+
|       1|      67885|2025-08-22 13:25:34|completed|          Jordan|
|       2|      54471|2025-06-20 06:13:44|cancelled|          Kuwait|
|       3|      63557|2024-10-24 12:16:58|cancelled|             UAE|
|       4|      78472|2024-12-19 22:00:33|  pending|             UAE|
|       5|      81175|2025-12-05 09:23:11|  shipped|           Qatar|
+--------+-----------+-------------------+---------+----------------+



In [21]:
#Read the order_items table using JDBC with an appropriate fetch size and partitioning.
order_items = spark.read \
    .format("jdbc") \
    .option("url", DB_URL) \
    .option("dbtable", "order_items") \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", DB_DRIVER) \
    .option("fetchsize", "10000") \
    .option("partitionColumn", "order_id") \
    .option("lowerBound", "1") \
    .option("upperBound", "500000") \
    .option("numPartitions", "8") \
    .load()

In [22]:
order_items.printSchema()

root
 |-- order_item_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)



In [23]:
order_items.createOrReplaceTempView("order_items")

In [24]:
spark.sql("""
SELECT *
FROM order_items
LIMIT 5
""").show()

+-------------+--------+----------+--------+----------+
|order_item_id|order_id|product_id|quantity|unit_price|
+-------------+--------+----------+--------+----------+
|            9|   25232|     14983|       2|    447.18|
|           14|    9551|       460|       1|   1949.69|
|           22|   17459|      8135|       3|    263.97|
|           40|   33568|     15476|       5|    976.07|
|           42|    3873|     10662|       4|     16.24|
+-------------+--------+----------+--------+----------+



In [25]:
#Read the payments table using JDBC with an appropriate fetch size and partitioning.
payments = spark.read \
    .format("jdbc") \
    .option("url", DB_URL) \
    .option("dbtable", "payments") \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", DB_DRIVER) \
    .option("fetchsize", "10000") \
    .option("partitionColumn", "payment_id") \
    .option("lowerBound", "1") \
    .option("upperBound", "500000") \
    .option("numPartitions", "4") \
    .load()

In [26]:
payments.printSchema()

root
 |-- payment_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- payment_date: timestamp (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)



In [27]:
payments.createOrReplaceTempView("payments")

In [28]:
spark.sql("""
SELECT *
FROM payments
LIMIT 5
""").show()

+----------+--------+--------------+--------------+-------------------+-------+
|payment_id|order_id|payment_method|payment_status|       payment_date| amount|
+----------+--------+--------------+--------------+-------------------+-------+
|         1|       1|          cash|          paid|2026-04-02 06:54:57| 358.38|
|         2|       2|        paypal|          paid|2025-03-12 15:28:26|4672.11|
|         3|       3|          cash|          paid|2026-04-21 10:19:09|1968.89|
|         4|       4| bank_transfer|      refunded|2025-09-17 12:33:49|3829.42|
|         5|       5|    debit_card|          paid|2026-01-25 11:11:09|2885.69|
+----------+--------+--------------+--------------+-------------------+-------+



In [69]:
#import functions that we will use
from pyspark.sql.functions import (
    col,
    trim,
    to_date,
    to_timestamp,
    when,
    sum,
    avg,
    count,
    concat_ws,
    lit,
    round,
    countDistinct,
    year,
    month,
    dayofmonth,
    current_timestamp,
    date_format
)

In [30]:
# Clean Customers
customers_clean = customers \
    .dropDuplicates(["customer_id"]) \
    .dropna(subset=[
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "country",
        "city",
        "signup_date"
    ])

In [31]:
# Clean Products
products_clean = products \
    .dropDuplicates(["product_id"]) \
    .dropna(subset=[
        "product_id",
        "product_name",
        "category",
        "price",
        "stock_quantity"
    ])

In [32]:
# Clean Orders
orders_clean = orders \
    .dropDuplicates(["order_id"]) \
    .dropna(subset=[
        "order_id",
        "customer_id",
        "order_date",
        "status",
        "shipping_country"
    ])

In [33]:
# Clean Order Items
order_items_clean = order_items \
    .dropDuplicates(["order_item_id"]) \
    .dropna(subset=[
        "order_item_id",
        "order_id",
        "product_id",
        "quantity",
        "unit_price"
    ])

In [34]:
# Clean Payments
payments_clean = payments \
    .dropDuplicates(["payment_id"]) \
    .dropna(subset=[
        "payment_id",
        "order_id",
        "payment_method",
        "payment_status",
        "payment_date",
        "amount"
    ])

Validate Dates

In [36]:
orders_clean = orders_clean.filter(
    col("order_date").isNotNull()
)

In [37]:
customers_clean = customers_clean.filter(
    col("signup_date").isNotNull()
)

In [38]:
payments_clean = payments_clean.filter(
    col("payment_date").isNotNull()
)

Validate Prices & Quantities

In [39]:
products_clean = products_clean.filter(
    col("price") > 0
)

In [40]:
order_items_clean = order_items_clean.filter(
    (col("quantity") > 0) &
    (col("unit_price") > 0)
)

In [41]:
payments_clean = payments_clean.filter(
    col("amount") > 0
)

Business Transformations

### Sales Table
Join orders, order_items, products, and customers to create the sales dataset
and calculate the required business metrics.

In [68]:
sales = orders_clean \
    .join(
        order_items_clean,
        orders_clean.order_id == order_items_clean.order_id,
        "inner"
    ) \
    .join(
        products_clean,
        order_items_clean.product_id == products_clean.product_id,
        "inner"
    ) \
    .join(
        customers_clean,
        orders_clean.customer_id == customers_clean.customer_id,
        "inner"
    ) \
    .select(
        orders_clean.order_id,
        orders_clean.customer_id,
        orders_clean.order_date,
        orders_clean.status,
        order_items_clean.product_id,
        order_items_clean.quantity,
        order_items_clean.unit_price,
        products_clean.product_name,
        customers_clean.first_name,
        customers_clean.last_name,
        products_clean.category,
        customers_clean.country
    ) \
    .withColumn(
        "total_sales",
        col("quantity") * col("unit_price")
    )

sales.show(5, False)

# Cache the sales dataset: every aggregation below reuses it,
# so Spark keeps it in memory instead of re-running the joins
# and re-reading MariaDB for each action.
sales = sales.cache()

+--------+-----------+-------------------+---------+----------+--------+----------+---------------------------------------------------+----------+---------+-----------+-------+-----------+
|order_id|customer_id|order_date         |status   |product_id|quantity|unit_price|product_name                                       |first_name|last_name|category   |country|total_sales|
+--------+-----------+-------------------+---------+----------+--------+----------+---------------------------------------------------+----------+---------+-----------+-------+-----------+
|347666  |26         |2025-01-13 12:17:52|completed|9852      |4       |306.70    |Multi-layered encompassing success                 |Jon       |Pacheco  |Accessories|Jordan |1226.80    |
|387046  |26         |2025-09-30 17:51:16|cancelled|13008     |5       |711.72    |Customer-focused intangible artificial intelligence|Jon       |Pacheco  |Clothing   |Jordan |3558.60    |
|449956  |26         |2026-05-13 22:11:24|completed|163

In [49]:
#Calculate total_sales
total_sales = sales.agg(
    sum("total_sales").alias("total_sales")
)

total_sales.show()

+-------------+
|  total_sales|
+-------------+
|6019380649.26|
+-------------+



In [51]:
#Total Quantity Sold Calculation

total_quantity = sales.agg(
    sum("quantity").alias("total_quantity_sold")
)

total_quantity.show()

+-------------------+
|total_quantity_sold|
+-------------------+
|            5998053|
+-------------------+



In [52]:
#Calculate the average value of an order.

aov = sales.agg(
    (
        sum("total_sales") / countDistinct("order_id")
    ).alias("average_order_value")
)

aov.show()

+-------------------+
|average_order_value|
+-------------------+
|    12259.505962888|
+-------------------+



In [53]:
#Calculate the number of unique orders.

number_of_orders = sales.agg(
    countDistinct("order_id").alias("number_of_orders")
)

number_of_orders.show()

+----------------+
|number_of_orders|
+----------------+
|          490997|
+----------------+



In [54]:
#Calculate the number of unique customers

number_of_customers = sales.agg(
    countDistinct("customer_id").alias("number_of_customers")
)

number_of_customers.show()

+-------------------+
|number_of_customers|
+-------------------+
|              99314|
+-------------------+



In [55]:
#Calculate total sales for each country

sales_by_country = sales.groupBy("country") \
    .agg(
        sum("total_sales").alias("total_sales")
    ) \
    .orderBy(col("total_sales").desc())

sales_by_country.show()

+------------+-------------+
|     country|  total_sales|
+------------+-------------+
|Saudi Arabia|1012751496.54|
|       Qatar|1010172854.31|
|         UAE|1008921516.54|
|      Kuwait|1004580220.46|
|       Egypt| 994959436.41|
|      Jordan| 987995125.00|
+------------+-------------+



In [56]:
#Calculate total sales for each product category.

sales_by_category = sales.groupBy("category") \
    .agg(
        sum("total_sales").alias("total_sales")
    ) \
    .orderBy(col("total_sales").desc())

sales_by_category.show()

+-----------+------------+
|   category| total_sales|
+-----------+------------+
|      Books|613007152.29|
|   Clothing|611584791.17|
|      Shoes|611322243.77|
|  Furniture|609126345.43|
|     Sports|607451054.31|
|     Beauty|604966211.58|
|Accessories|593922059.17|
|     Gaming|590665869.72|
|       Home|590375606.82|
|Electronics|586959315.00|
+-----------+------------+



In [58]:
#Calculate total sales for each day
sales_by_day = sales \
    .withColumn("sale_day", to_date("order_date")) \
    .groupBy("sale_day") \
    .agg(
        sum("total_sales").alias("total_sales")
    ) \
    .orderBy("sale_day")

sales_by_day.show()

+----------+-----------+
|  sale_day|total_sales|
+----------+-----------+
|2024-09-08| 5326784.45|
|2024-09-09| 8358873.31|
|2024-09-10| 8399754.12|
|2024-09-11| 8263076.49|
|2024-09-12| 9283841.23|
|2024-09-13| 8474702.54|
|2024-09-14| 8090763.53|
|2024-09-15| 8768590.49|
|2024-09-16| 8567857.21|
|2024-09-17| 8196453.97|
|2024-09-18| 8008784.23|
|2024-09-19| 8248600.12|
|2024-09-20| 8065315.84|
|2024-09-21| 8527447.59|
|2024-09-22| 7766541.30|
|2024-09-23| 7652686.47|
|2024-09-24| 7539975.80|
|2024-09-25| 7725625.92|
|2024-09-26| 7841607.82|
|2024-09-27| 8391033.87|
+----------+-----------+
only showing top 20 rows



In [59]:
#Calculate total sales for each month
sales_by_month = sales \
    .withColumn(
        "sale_month",
        date_format("order_date", "yyyy-MM")
    ) \
    .groupBy("sale_month") \
    .agg(
        sum("total_sales").alias("total_sales")
    ) \
    .orderBy("sale_month")

sales_by_month.show(5)

+----------+------------+
|sale_month| total_sales|
+----------+------------+
|   2024-09|186588447.35|
|   2024-10|253322302.21|
|   2024-11|249259779.25|
|   2024-12|258816866.62|
|   2025-01|253688049.63|
+----------+------------+
only showing top 5 rows



In [60]:
#Calculate total sales for each year

sales_by_year = sales \
    .withColumn(
        "sale_year",
        date_format("order_date", "yyyy")
    ) \
    .groupBy("sale_year") \
    .agg(
        sum("total_sales").alias("total_sales")
    ) \
    .orderBy("sale_year")

sales_by_year.show(5)

+---------+-------------+
|sale_year|  total_sales|
+---------+-------------+
|     2024| 947987395.43|
|     2025|3017298101.54|
|     2026|2054095152.29|
+---------+-------------+



Daily Sales
Aggregate the existing sales data by year, month, and day, using the previously calculated total sales to derive daily sales metrics.

In [61]:
daily_sales = sales \
    .withColumn("year", year("order_date")) \
    .withColumn("month", month("order_date")) \
    .withColumn("day", dayofmonth("order_date")) \
    .groupBy("year", "month", "day") \
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        sum("total_sales").alias("total_sales")
    ) \
    .withColumn(
        "average_order_value",
        round(col("total_sales") / col("total_orders"), 2)
    ) \
    .orderBy("year", "month", "day")

daily_sales.show()

+----+-----+---+------------+--------------+-----------+-------------------+
|year|month|day|total_orders|total_quantity|total_sales|average_order_value|
+----+-----+---+------------+--------------+-----------+-------------------+
|2024|    9|  8|         447|          5475| 5326784.45|           11916.74|
|2024|    9|  9|         690|          8265| 8358873.31|           12114.31|
|2024|    9| 10|         707|          8439| 8399754.12|           11880.84|
|2024|    9| 11|         691|          8337| 8263076.49|           11958.14|
|2024|    9| 12|         738|          9256| 9283841.23|           12579.73|
|2024|    9| 13|         699|          8527| 8474702.54|           12124.04|
|2024|    9| 14|         670|          8102| 8090763.53|           12075.77|
|2024|    9| 15|         705|          8686| 8768590.49|           12437.72|
|2024|    9| 16|         686|          8572| 8567857.21|           12489.59|
|2024|    9| 17|         679|          8141| 8196453.97|           12071.36|

### Customer Sales

In [70]:
customer_sales = sales \
    .withColumn(
        "customer_name",
        concat_ws(" ", col("first_name"), col("last_name"))
    ) \
    .groupBy(
        "customer_id",
        "customer_name",
        "country"
    ) \
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        sum("total_sales").alias("total_spending")
    ) \
    .orderBy("customer_id")

customer_sales.show(10, False)

+-----------+---------------+-------+------------+--------------+--------------+
|customer_id|customer_name  |country|total_orders|total_quantity|total_spending|
+-----------+---------------+-------+------------+--------------+--------------+
|1          |Robin Heath    |Kuwait |3           |31            |33624.57      |
|2          |Sarah Long     |UAE    |4           |38            |46206.38      |
|3          |Lori Green     |Qatar  |5           |61            |58523.64      |
|4          |April Sexton   |Kuwait |5           |51            |54357.45      |
|5          |Andrew Flores  |Kuwait |9           |101           |115123.60     |
|6          |Eric Edwards   |Jordan |6           |65            |78117.54      |
|7          |Daniel Saunders|Egypt  |3           |35            |43263.79      |
|8          |Maria Scott    |Egypt  |4           |29            |34539.60      |
|9          |Cheryl Peterson|Qatar  |10          |136           |154154.28     |
|10         |Edward Blake   

### Product Sales

In [71]:
product_sales = sales \
    .groupBy(
        "product_id",
        "product_name",
        "category"
    ) \
    .agg(
        sum("quantity").alias("total_quantity"),
        sum("total_sales").alias("total_sales")
    ) \
    .orderBy("product_id")

product_sales.show(10, False)

+----------+-------------------------------------------+-----------+--------------+-----------+
|product_id|product_name                               |category   |total_quantity|total_sales|
+----------+-------------------------------------------+-----------+--------------+-----------+
|1         |Decentralized explicit solution            |Home       |297           |344646.41  |
|2         |Up-sized 3rdgeneration system engine       |Beauty     |304           |318345.50  |
|3         |Open-source tertiary complexity            |Accessories|263           |274460.16  |
|4         |Seamless holistic policy                   |Books      |336           |335869.50  |
|5         |Reactive contextually-based functionalities|Clothing   |298           |295009.94  |
|6         |Assimilated solution-oriented hierarchy    |Books      |329           |309465.09  |
|7         |Programmable real-time extranet            |Shoes      |272           |251200.99  |
|8         |Optional composite function 

In [73]:
spark.sql("CREATE DATABASE IF NOT EXISTS ecommerce_dw")

2026-09-10 01:27:15,730 WARN metastore.ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
2026-09-10 01:27:15,887 WARN metastore.ObjectStore: Failed to get database ecommerce_dw, returning NoSuchObjectException


DataFrame[]

In [75]:
sales.write.mode("overwrite").saveAsTable("ecommerce_dw.fact_sales")

daily_sales.write.mode("overwrite").saveAsTable("ecommerce_dw.daily_sales")

customer_sales.write.mode("overwrite").saveAsTable("ecommerce_dw.customer_sales")

product_sales.write.mode("overwrite").saveAsTable("ecommerce_dw.product_sales")

2026-09-10 01:31:08,065 WARN session.SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
2026-09-10 01:31:08,965 WARN conf.HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
2026-09-10 01:31:08,965 WARN conf.HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
2026-09-10 01:31:08,966 WARN conf.HiveConf: HiveConf of name hive.stats.retries.wait does not exist


In [80]:
spark.sql("SHOW TABLES IN ecommerce_dw").show()

+------------+--------------+-----------+
|    database|     tableName|isTemporary|
+------------+--------------+-----------+
|ecommerce_dw|customer_sales|      false|
|ecommerce_dw|   daily_sales|      false|
|ecommerce_dw|    fact_sales|      false|
|ecommerce_dw| product_sales|      false|
|            |     customers|       true|
|            |   order_items|       true|
|            |        orders|       true|
|            |      payments|       true|
|            |      products|       true|
+------------+--------------+-----------+

